# 02. Tratamiento de datos epidemiológicos

## Objetivo
Preparar los registros históricos de vigilancia epidemiológica
del dengue en Perú para su posterior análisis mediante
Isolation Forest.

## Procedimiento
1. Cargar el dataset original.
2. Estandarizar variables.
3. Validar la consistencia de los registros.
4. Agrupar casos por distrito y semana epidemiológica.
5. Exportar el dataset tratado.

**Unidad de análisis:** UBIGEO + año + semana epidemiológica.

## 1. Cargar el dataset

In [2]:
from pathlib import Path
import pandas as pd

NOMBRE_CSV = "datos_abiertos_vigilancia_dengue_2000_2024.csv"
PROJECT_ROOT = next(
    (
        carpeta
        for carpeta in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (carpeta / "data" / "dataset" / NOMBRE_CSV).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"No se encontró data/dataset/{NOMBRE_CSV}. "
        "Coloca allí el CSV y ejecuta esta celda desde el proyecto."
    )

DATA_PATH = PROJECT_ROOT / "data" / "dataset" / NOMBRE_CSV

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False,
    dtype={"diresa": "string", "localidad": "string", "ubigeo": "string", "localcod": "string"},
)

print("Archivo cargado:", DATA_PATH.name)
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

display(df.head())

Archivo cargado: datos_abiertos_vigilancia_dengue_2000_2024.csv
Filas: 1,029,421
Columnas: 14


,departamento,provincia,distrito,localidad,enfermedad,ano,semana,diagnostic,diresa,ubigeo,localcod,edad,tipo_edad,sexo
0,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,<NA>,58,A,M
1,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,<NA>,44,A,F
2,LORETO,ALTO AMAZONAS,YURIMAGUAS,BARRIO MORALILLOS,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,<NA>,21,A,F
3,LORETO,MAYNAS,IQUITOS,005902,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,<NA>,74,A,M
4,LORETO,MAYNAS,IQUITOS,CARDOZO,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,<NA>,42,A,F


## 2. Estandarizar variables

In [3]:
df_limpio = df.copy()

columnas_texto = [
    "departamento",
    "provincia",
    "distrito",
    "enfermedad",
    "diagnostic",
    "sexo",
    "tipo_edad"
]

for col in columnas_texto:
    df_limpio[col] = (
        df_limpio[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

df_limpio["ubigeo"] = (
    df_limpio["ubigeo"]
    .astype("string")
    .str.strip()
)

df_limpio["ano"] = pd.to_numeric(
    df_limpio["ano"], errors="coerce"
)

df_limpio["semana"] = pd.to_numeric(
    df_limpio["semana"], errors="coerce"
)

print("Estandarización completada")

Estandarización completada


## 3. Validar registros

In [4]:
validos = (
    df_limpio["ano"].between(2000, 2024)
    & df_limpio["semana"].between(1, 53)
    & df_limpio["ubigeo"].str.fullmatch(r"\d{6}").fillna(False)
)

df_excluidos = df_limpio.loc[~validos].copy()
df_validado = df_limpio.loc[validos].copy()

print("Registros originales:", len(df_limpio))
print("Registros válidos:", len(df_validado))
print("Registros separados para revisión:", len(df_excluidos))

assert len(df_limpio) == len(df_validado) + len(df_excluidos)

display(df_excluidos.head())

Registros originales: 1029421
Registros válidos: 1029421
Registros separados para revisión: 0


,departamento,provincia,distrito,localidad,enfermedad,ano,semana,diagnostic,diresa,ubigeo,localcod,edad,tipo_edad,sexo


## 4. Añadir la información geográfica

In [12]:
# Crear variables epidemiológicas

df_validado["dengue_grave"] = (
    df_validado["enfermedad"] == "DENGUE GRAVE"
).astype(int)

df_validado["con_alarma"] = (
    df_validado["enfermedad"] == "DENGUE CON SIGNOS DE ALARMA"
).astype(int)

df_validado["sexo_f"] = (
    df_validado["sexo"] == "F"
).astype(int)

# Agrupar registros por distrito, año y semana

claves = ["ubigeo", "ano", "semana"]

df_semanal = (
    df_validado
    .groupby(claves, as_index=False)
    .agg(
        casos_semanales=("enfermedad", "size"),
        casos_graves=("dengue_grave", "sum"),
        casos_con_alarma=("con_alarma", "sum"),
        casos_femeninos=("sexo_f", "sum")
    )
)

# Calcular proporciones clínicas

df_semanal["proporcion_graves"] = (
    df_semanal["casos_graves"] /
    df_semanal["casos_semanales"]
)

df_semanal["proporcion_alarma"] = (
    df_semanal["casos_con_alarma"] /
    df_semanal["casos_semanales"]
)

print("Agrupación epidemiológica completada")
print("Observaciones semanales:", len(df_semanal))

display(df_semanal.head(10))

Agrupación epidemiológica completada
Observaciones semanales: 81196


,ubigeo,ano,semana,casos_semanales,casos_graves,casos_con_alarma,casos_femeninos,proporcion_graves,proporcion_alarma
0,010101,2004,39,1,0,0,1,0.0,0.0
1,010101,2006,40,2,0,0,1,0.0,0.0
2,010101,2007,50,1,0,0,1,0.0,0.0
3,010101,2011,4,1,0,0,0,0.0,0.0
4,010101,2015,9,1,0,0,1,0.0,0.0
5,010101,2015,34,1,0,0,0,0.0,0.0
6,010101,2015,45,1,0,0,0,0.0,0.0
7,010101,2018,26,1,0,0,1,0.0,0.0
8,010101,2019,50,1,0,1,0,0.0,1.0
9,010101,2022,20,1,0,0,1,0.0,0.0


In [14]:
if len(conflictos) == 0:
    df_semanal = df_semanal.merge(
        territorios,
        on="ubigeo",
        how="left",
        validate="many_to_one"
    )

    print("Información geográfica incorporada correctamente")
    display(df_semanal.head(10))

else:
    print("Revisar las inconsistencias territoriales.")

Información geográfica incorporada correctamente


,ubigeo,ano,semana,casos_semanales,casos_graves,casos_con_alarma,casos_femeninos,proporcion_graves,proporcion_alarma,departamento,provincia,distrito
0,010101,2004,39,1,0,0,1,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
1,010101,2006,40,2,0,0,1,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
2,010101,2007,50,1,0,0,1,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
3,010101,2011,4,1,0,0,0,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
4,010101,2015,9,1,0,0,1,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
5,010101,2015,34,1,0,0,0,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
6,010101,2015,45,1,0,0,0,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
7,010101,2018,26,1,0,0,1,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
8,010101,2019,50,1,0,1,0,0.0,1.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS
9,010101,2022,20,1,0,0,1,0.0,0.0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS


In [19]:
df_verificacion = pd.read_csv(
    archivo_salida,
    dtype={"ubigeo": "string"}
)

assert len(df_verificacion) == 81196
assert len(df_verificacion.columns) == 12

print("Archivo CSV verificado correctamente.")
print("Filas:", len(df_verificacion))
print("Columnas:", len(df_verificacion.columns))

NameError: name 'archivo_salida' is not defined

## 5. Exportar los resultados

Una vez verificadas la integridad y consistencia de los datos
agregados, se exporta el conjunto de observaciones
epidemiológicas semanales para su posterior utilización
en el modelo Isolation Forest.

In [20]:
from pathlib import Path

# 1. Identificar la carpeta principal del proyecto
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# 2. Definir la carpeta de salida
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed"

# 3. Crear la carpeta si no existe
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# 4. Ordenar los datos epidemiológicos
df_semanal = df_semanal.sort_values(
    by=["departamento", "provincia", "distrito", "ano", "semana"]
).reset_index(drop=True)

# 5. Definir el archivo CSV
archivo_salida = OUTPUT_PATH / "dengue_distrito_semanal.csv"

# 6. Exportar los datos procesados
df_semanal.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

# 7. Mostrar los resultados
print("=" * 55)
print("EXPORTACIÓN COMPLETADA")
print("=" * 55)

print("Archivo:", archivo_salida)
print("Observaciones:", len(df_semanal))
print("Variables:", len(df_semanal.columns))

print(
    "Tamaño (MB):",
    round(archivo_salida.stat().st_size / 1024**2, 2)
)

print("\nPrimeras observaciones:")
display(df_semanal.head(10))

EXPORTACIÓN COMPLETADA
Archivo: D:\UNSCH\IA I\Trabajo Final\isolation-forest-dengue-peru\data\processed\dengue_distrito_semanal.csv
Observaciones: 81196
Variables: 12
Tamaño (MB): 4.88

Primeras observaciones:


,ubigeo,ano,semana,casos_semanales,casos_graves,casos_con_alarma,casos_femeninos,proporcion_graves,proporcion_alarma,departamento,provincia,distrito
0,010202,2001,13,2,0,0,0,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
1,010202,2001,14,2,0,0,2,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
2,010202,2001,15,8,0,0,8,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
3,010202,2001,16,4,0,0,2,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
4,010202,2001,17,3,0,0,3,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
5,010202,2001,20,2,0,0,2,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
6,010202,2001,35,1,0,0,0,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
7,010202,2003,8,1,0,0,0,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
8,010202,2004,1,1,0,0,1,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
9,010202,2004,18,1,0,0,1,0.0,0.0,AMAZONAS,BAGUA,ARAMANGO
